In [2]:
!pip install pyspark

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, sum, count

In [4]:
spark = SparkSession.builder \
    .appName("PySpark Assignment") \
    .getOrCreate()

print("Spark started successfully!")

Spark started successfully!


In [5]:
%%writefile sales.csv
Product,Category,Sales,Quantity
Laptop,Electronics,1000,2
Phone,Electronics,500,5
Chair,Furniture,200,4
Table,Furniture,300,2
Laptop,Electronics,1000,2
Headphones,Electronics,150,10
Chair,Furniture,200,4
Book,Stationery,50,20

Writing sales.csv


In [6]:
df = spark.read.csv("sales.csv", header=True, inferSchema=True)

df.show()

print("Schema:")
df.printSchema()

+----------+-----------+-----+--------+
|   Product|   Category|Sales|Quantity|
+----------+-----------+-----+--------+
|    Laptop|Electronics| 1000|       2|
|     Phone|Electronics|  500|       5|
|     Chair|  Furniture|  200|       4|
|     Table|  Furniture|  300|       2|
|    Laptop|Electronics| 1000|       2|
|Headphones|Electronics|  150|      10|
|     Chair|  Furniture|  200|       4|
|      Book| Stationery|   50|      20|
+----------+-----------+-----+--------+

Schema:
root
 |-- Product: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sales: integer (nullable = true)
 |-- Quantity: integer (nullable = true)



In [7]:
filtered_df = df.filter(col("Sales") > 500)

filtered_df.show()

+-------+-----------+-----+--------+
|Product|   Category|Sales|Quantity|
+-------+-----------+-----+--------+
| Laptop|Electronics| 1000|       2|
| Laptop|Electronics| 1000|       2|
+-------+-----------+-----+--------+



In [8]:
df.filter(df.Sales > 500).show()

+-------+-----------+-----+--------+
|Product|   Category|Sales|Quantity|
+-------+-----------+-----+--------+
| Laptop|Electronics| 1000|       2|
| Laptop|Electronics| 1000|       2|
+-------+-----------+-----+--------+



In [9]:
grouped_df = df.groupBy("Category").agg(
    sum("Sales").alias("Total_Sales")
)

grouped_df.show()

+-----------+-----------+
|   Category|Total_Sales|
+-----------+-----------+
| Stationery|         50|
|Electronics|       2650|
|  Furniture|        700|
+-----------+-----------+



In [10]:
grouped_df = df.groupBy("Category").sum("Sales")

grouped_df.show()

+-----------+----------+
|   Category|sum(Sales)|
+-----------+----------+
| Stationery|        50|
|Electronics|      2650|
|  Furniture|       700|
+-----------+----------+



In [11]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, sum

spark = SparkSession.builder \
    .appName("Sales Analysis") \
    .getOrCreate()

df = spark.read.csv(
    "sales.csv",
    header=True,
    inferSchema=True
)

df.show()

+----------+-----------+-----+--------+
|   Product|   Category|Sales|Quantity|
+----------+-----------+-----+--------+
|    Laptop|Electronics| 1000|       2|
|     Phone|Electronics|  500|       5|
|     Chair|  Furniture|  200|       4|
|     Table|  Furniture|  300|       2|
|    Laptop|Electronics| 1000|       2|
|Headphones|Electronics|  150|      10|
|     Chair|  Furniture|  200|       4|
|      Book| Stationery|   50|      20|
+----------+-----------+-----+--------+



In [12]:
from pyspark.sql.functions import sum

df.groupBy("Category").agg(
    sum("Sales").alias("Total_Sales")
).show()

+-----------+-----------+
|   Category|Total_Sales|
+-----------+-----------+
| Stationery|         50|
|Electronics|       2650|
|  Furniture|        700|
+-----------+-----------+



In [13]:
df.dropDuplicates().show()

+----------+-----------+-----+--------+
|   Product|   Category|Sales|Quantity|
+----------+-----------+-----+--------+
|      Book| Stationery|   50|      20|
|    Laptop|Electronics| 1000|       2|
|     Table|  Furniture|  300|       2|
|     Phone|Electronics|  500|       5|
|     Chair|  Furniture|  200|       4|
|Headphones|Electronics|  150|      10|
+----------+-----------+-----+--------+



In [14]:
from pyspark.sql.functions import sum, avg

grouped_df = df.groupBy("Category").agg(
    sum("Sales").alias("Total_Sales")
)

print("Total Sales by Category:")
grouped_df.show()

Total Sales by Category:
+-----------+-----------+
|   Category|Total_Sales|
+-----------+-----------+
| Stationery|         50|
|Electronics|       2650|
|  Furniture|        700|
+-----------+-----------+



In [15]:
deduplicated_df = df.dropDuplicates()

print("Dataset after removing duplicates:")
deduplicated_df.show()

Dataset after removing duplicates:
+----------+-----------+-----+--------+
|   Product|   Category|Sales|Quantity|
+----------+-----------+-----+--------+
|      Book| Stationery|   50|      20|
|    Laptop|Electronics| 1000|       2|
|     Table|  Furniture|  300|       2|
|     Phone|Electronics|  500|       5|
|     Chair|  Furniture|  200|       4|
|Headphones|Electronics|  150|      10|
+----------+-----------+-----+--------+



In [16]:
%%writefile product_info.csv
Product,Brand
Laptop,Dell
Phone,Samsung
Chair,IKEA
Table,IKEA
Headphones,Sony
Book,Penguin

Writing product_info.csv


In [17]:
product_df = spark.read.csv(
    "product_info.csv",
    header=True,
    inferSchema=True
)

product_df.show()

+----------+-------+
|   Product|  Brand|
+----------+-------+
|    Laptop|   Dell|
|     Phone|Samsung|
|     Chair|   IKEA|
|     Table|   IKEA|
|Headphones|   Sony|
|      Book|Penguin|
+----------+-------+



In [18]:
joined_df = df.join(
    product_df,
    on="Product",
    how="inner"
)

print("Joined Dataset:")
joined_df.show()

Joined Dataset:
+----------+-----------+-----+--------+-------+
|   Product|   Category|Sales|Quantity|  Brand|
+----------+-----------+-----+--------+-------+
|    Laptop|Electronics| 1000|       2|   Dell|
|     Phone|Electronics|  500|       5|Samsung|
|     Chair|  Furniture|  200|       4|   IKEA|
|     Table|  Furniture|  300|       2|   IKEA|
|    Laptop|Electronics| 1000|       2|   Dell|
|Headphones|Electronics|  150|      10|   Sony|
|     Chair|  Furniture|  200|       4|   IKEA|
|      Book| Stationery|   50|      20|Penguin|
+----------+-----------+-----+--------+-------+



In [19]:
average_sales = df.groupBy("Category").agg(
    avg("Sales").alias("Average_Sales")
)

print("Average Sales by Product Category:")
average_sales.show()

Average Sales by Product Category:
+-----------+------------------+
|   Category|     Average_Sales|
+-----------+------------------+
| Stationery|              50.0|
|Electronics|             662.5|
|  Furniture|233.33333333333334|
+-----------+------------------+

